In [ ]:
# ☁️ Serverless Computing Architecture & Deployment Guide

**Building scalable, cost-effective applications without managing servers**

This guide covers:
- AWS Lambda, Azure Functions, Google Cloud Functions
- Cold starts and warm-up strategies
- State management in serverless
- Cost optimization
- Monitoring and debugging
- Security in serverless
- Framework selection (SAM, Serverless Framework, FastAPI)

## 1. Serverless Platforms

### AWS Lambda
```yaml
# SAM template (serverless.yaml alternative)
AWSTemplateFormatVersion: '2010-09-09'
Transform: AWS::Serverless-2016-10-31

Resources:
  ProcessOrderFunction:
    Type: AWS::Serverless::Function
    Properties:
      Runtime: python3.11
      Handler: app.lambda_handler
      Timeout: 60
      Memory: 512
      Environment:
        Variables:
          DB_HOST: !Ref DBCluster
          LOG_LEVEL: INFO
      Events:
        ApiEvent:
          Type: Api
          Properties:
            Path: /orders
            Method: POST
            RestApiId: !Ref OrderApi
      Policies:
        - DynamoDBCrudPolicy:
            TableName: !Ref OrdersTable
        - CloudWatchPutMetricPolicy: {}
```

```python
# Lambda handler
import json
import logging
import boto3
from datetime import datetime

logger = logging.getLogger()
logger.setLevel(logging.INFO)

dynamodb = boto3.resource('dynamodb')
orders_table = dynamodb.Table('orders')

def lambda_handler(event, context):
    """Process incoming order"""
    try:
        # Parse event
        body = json.loads(event['body'])

        # Validate
        if not body.get('items'):
            return {
                'statusCode': 400,
                'body': json.dumps({'error': 'No items'})
            }

        # Process order
        order_id = str(datetime.now().timestamp())
        total = sum(item['price'] * item['quantity'] for item in body['items'])

        # Store in DynamoDB
        orders_table.put_item(Item={
            'order_id': order_id,
            'user_id': body['user_id'],
            'items': body['items'],
            'total': total,
            'created_at': datetime.now().isoformat()
        })

        logger.info(f"Order created: {order_id}")

        return {
            'statusCode': 201,
            'body': json.dumps({
                'order_id': order_id,
                'total': total
            })
        }

    except Exception as e:
        logger.error(f"Error: {str(e)}")
        return {
            'statusCode': 500,
            'body': json.dumps({'error': 'Internal server error'})
        }
```

### Azure Functions
```python
# Azure Functions (Python)
import os
import json
from datetime import datetime
import azure.functions as func
from azure.data.tables import TableClient

def main(req: func.HttpRequest) -> func.HttpResponse:
    """HTTP trigger function"""

    try:
        body = req.get_json()

        # Access connection from environment
        table_client = TableClient.from_connection_string(
            os.environ['AzureWebJobsStorage'],
            'orders'
        )

        # Create entity
        entity = {
            'PartitionKey': body['user_id'],
            'RowKey': str(datetime.now().timestamp()),
            'items': json.dumps(body['items']),
            'total': sum(i['price'] * i['qty'] for i in body['items'])
        }

        table_client.create_entity(entity)

        return func.HttpResponse(
            json.dumps({'status': 'created'}),
            status_code=201
        )

    except Exception as e:
        return func.HttpResponse(
            json.dumps({'error': str(e)}),
            status_code=500
        )
```

## 2. Cold Start Optimization

### Problem: Cold Start Latency
```
Without optimization:
Request 1: ~2-5 seconds (cold start)
Requests 2-N: ~100-500ms (warm)

With provisioned concurrency/reserved capacity:
All requests: ~100-500ms (always warm)
```

### Strategies

```python
# 1. Minimize package size
# Remove unnecessary dependencies
def create_deployment_package():
    """Create optimized package"""
    import subprocess

    # Install only production dependencies
    subprocess.run([
        'pip', 'install',
        '-r', 'requirements.txt',
        '-t', 'package/',
        '--no-cache-dir'
    ])

    # Remove unused files
    for pattern in ['*.dist-info', '*.pyc', '__pycache__', 'tests']:
        subprocess.run(['find', 'package', '-name', pattern, '-type', 'd', '-exec', 'rm', '-rf', '{}', '+'])

    # Create zip (only 5-20MB instead of 100MB+)
    subprocess.run(['zip', '-r', 'function.zip', 'package/', 'app.py'])

# 2. Use Lambda layers for dependencies
# 3. Keep code and imports minimal
# 4. Use provisioned concurrency (AWS)
# 5. Use reserved instances (Azure)
```

### Warm-up Orchestration
```python
import asyncio
import httpx

async def warm_up_lambda(lambda_name: str, count: int = 5):
    """Pre-warm Lambda functions"""
    async with httpx.AsyncClient() as client:
        tasks = []

        for i in range(count):
            url = f"https://lambda.amazonaws.com/warm-up/{i}"
            tasks.append(client.post(url))

        results = await asyncio.gather(*tasks)

        warmed = sum(1 for r in results if r.status_code == 200)
        print(f"Warmed {warmed}/{count} instances")
```

## 3. State Management in Serverless

### External State Stores
```python
# Each function is stateless; store state externally

# Option 1: DynamoDB/CosmosDB
import json
import time
from datetime import datetime
import boto3

dynamodb = boto3.resource('dynamodb')
state_table = dynamodb.Table('function_state')

def save_state(request_id, data, ttl=3600):
    state_table.put_item(Item={
        'request_id': request_id,
        'data': data,
        'ttl': int(time.time()) + ttl,
        'created_at': datetime.now().isoformat()
    })

def get_state(request_id):
    response = state_table.get_item(Key={'request_id': request_id})
    return response.get('Item', {}).get('data')

# Option 2: Redis
import redis

r = redis.Redis(host='cache.amazonaws.com', decode_responses=True)

def save_state_redis(request_id, data):
    r.setex(request_id, 3600, json.dumps(data))

def get_state_redis(request_id):
    data = r.get(request_id)
    return json.loads(data) if data else None
```

### Long-Running Operations (Step Functions)
```python
import json
import boto3

stepfunctions = boto3.client('stepfunctions')

def start_workflow(definition):
    """Start Step Function execution"""
    response = stepfunctions.start_execution(
        stateMachineArn='arn:aws:states:...',
        input=json.dumps({
            'orderId': '12345',
            'items': [...]
        })
    )

    return response['executionArn']

# YAML definition of workflow
definition = """
StartAt: ValidateOrder
States:
  ValidateOrder:
    Type: Task
    Resource: arn:aws:lambda:...:function:validate-order
    Next: ProcessPayment
  ProcessPayment:
    Type: Task
    Resource: arn:aws:lambda:...:function:process-payment
    Next: ShipOrder
  ShipOrder:
    Type: Task
    Resource: arn:aws:lambda:...:function:ship-order
    End: true
"""
```

## 4. Cost Optimization & Monitoring

### Cost Calculation
```
AWS Lambda pricing:
- Requests: $0.20 per 1 million requests
- Compute: $0.0000166667 per GB-second
  (1GB memory × 1 second = $0.0000166667)

Example:
- 1M requests/month
- 256MB memory (0.25 GB)
- 2 second average duration

Monthly cost = (1M × $0.20 / 1M) + (1M × 2s × 0.25GB × $0.0000166667)
             = $0.20 + $8.33
             = ~$8.53/month
```

### Monitoring
```python
import json
from aws_lambda_powertools import Logger, Tracer, Metrics

logger = Logger()
tracer = Tracer()
metrics = Metrics()

@tracer.capture_lambda_handler
def lambda_handler(event, context):
    # Log structured data
    logger.info("Processing order", extra={
        "order_id": event['orderId'],
        "user_id": event['userId']
    })

    # Track metrics
    metrics.add_metric(
        name="OrderProcessed",
        unit="Count",
        value=1
    )

    # X-Ray tracing
    with tracer.capture_subsegment("payment_processing"):
        process_payment(event)

    return {
        'statusCode': 200,
        'body': json.dumps({'success': True})
    }
```

## 5. Serverless Best Practices Checklist

✅ **Architecture**
- [ ] Stateless functions (store state externally)
- [ ] Event-driven design
- [ ] Proper function boundaries
- [ ] DLQ for failed events

✅ **Performance**
- [ ] Minimize cold starts (package size, provisioned concurrency)
- [ ] Connection pooling for databases
- [ ] Caching strategies
- [ ] Async operations where possible

✅ **Security**
- [ ] IAM roles with least privilege
- [ ] Secrets management (Secrets Manager, Key Vault)
- [ ] VPC configuration if needed
- [ ] Input validation

✅ **Monitoring**
- [ ] Structured logging
- [ ] Metrics and alarms
- [ ] Distributed tracing (X-Ray, Azure Monitor)
- [ ] Cost tracking

SyntaxError: invalid decimal literal (910315752.py, line 157)